<a href="https://colab.research.google.com/github/camille2019/Women-Capital-Trial-Analysis/blob/main/propensity_matching_psm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [29]:
!pip install psmpy


In [30]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

from psmpy import PsmPy
from psmpy.functions import cohenD
from psmpy.plotting import *

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
save_path = '/content/drive/MyDrive/Capital_trials/'
full_meta = pd.read_csv(save_path + 'full_crime_meta_.csv')
ohe_meta = pd.read_csv('/content/drive/MyDrive/Capital_trials/ohe_metadata_only.csv')

In [43]:
def add_propensity(propensity_columns, control_column, treatment, df_encoded, df_meta, matched_group_name, cal = None):

  X = df_encoded[propensity_columns]
  T = df_encoded[control_column]


  psm_df =  X.copy()
  psm_df[treatment] = T.astype(int)

  psm_df = psm_df.reset_index(drop=True)
  psm_df["row_id"] = psm_df.index

  psm = PsmPy(psm_df, treatment=treatment, indx="row_id", exclude=[])

  psm.logistic_ps(balance=True)


  if cal is not None:
    psm.kdtree_matched(matcher='propensity_logit', replacement=False, caliper=cal)

  pred = psm.predicted_data

  df_meta = df_meta.reset_index(drop=True)

  scores_col_name = matched_group_name + '_prop_scores'

  df_meta[scores_col_name] = pred["propensity_score"].values

  psm.kdtree_matched(matcher="propensity_score", replacement=False, caliper=None)
  matched_df = psm.df_matched
  matched_ids = set(psm.df_matched["row_id"].unique())
  df_meta = df_meta.reset_index(drop=True)

  binary_col_name = 'psmpropensity_' + matched_group_name

  df_meta[binary_col_name] = df_meta.index.isin(matched_ids)

  return df_meta

In [22]:

state_race_crime_cols = ['State_AL', 'State_AZ', 'State_CA','State_FL', 'State_GA',
 'State_LA', 'State_MS', 'State_NC', 'State_OH',
 'State_OK', 'State_PA', 'State_SC', 'State_TN', 'State_TX',
  'Race/Ethnicity_Asian', 'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
   'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White',
                'Continuing threat to society', 'Disregard for human life', 'Manner - mayhem',
                'Multiple - harm to multiple people', 'Retaliation', 'Victim - old or disabled',
                'Weapon - arson', 'Weapon - deadly weapon', 'aggravated assault', 'attempted rape',
                'burglary/robbery', 'child victim', 'created public risk', 'deliberate and premeditated',
                'done while committing multiplie felonies', 'especially cruel', 'fiancial gain', 'financial gain',
                'financial gian', 'gang murder', 'intentional infliction of great bodily injury', 'interfere with prosecution/arrest',
                'kidnapping', 'larceny', 'lewd acts with a child', 'lying in wait', 'multiple murders',
                'murdered an officer in the line of duty', 'personal use of a handgun', 'poison', 'prior conviction',
                'prior offense', 'prior offenses', 'rape', 'robbery',
                'sexual assult', 'shooting at an occupied vehicle', 'sodomy', 'torture', 'unlawful possession of a firearm']



In [23]:
race_crime_cols = [ 'Race/Ethnicity_Asian', 'Race/Ethnicity_Black', 'Race/Ethnicity_Latino',
   'Race/Ethnicity_Latinx', 'Race/Ethnicity_Native American',
'Race/Ethnicity_Native American,White,Black', 'Race/Ethnicity_White',
                'Continuing threat to society', 'Disregard for human life', 'Manner - mayhem',
                'Multiple - harm to multiple people', 'Retaliation', 'Victim - old or disabled',
                'Weapon - arson', 'Weapon - deadly weapon', 'aggravated assault', 'attempted rape',
                'burglary/robbery', 'child victim', 'created public risk', 'deliberate and premeditated',
                'done while committing multiplie felonies', 'especially cruel', 'fiancial gain', 'financial gain',
                'financial gian', 'gang murder', 'intentional infliction of great bodily injury', 'interfere with prosecution/arrest',
                'kidnapping', 'larceny', 'lewd acts with a child', 'lying in wait', 'multiple murders',
                'murdered an officer in the line of duty', 'personal use of a handgun', 'poison', 'prior conviction',
                'prior offense', 'prior offenses', 'rape', 'robbery',
                'sexual assult', 'shooting at an occupied vehicle', 'sodomy', 'torture', 'unlawful possession of a firearm',]

In [45]:
updated_df = add_propensity(race_crime_cols, 'gender_women', 'gender', ohe_meta, full_meta, 'race_crime_match')
updated_df = add_propensity(state_race_crime_cols, 'gender_women', 'gender', ohe_meta, updated_df, 'state_race_crime_match')


In [46]:
#adjust caliper to allow for fuzzy matchng
updated_df = add_propensity(race_crime_cols, 'gender_women', 'gender', ohe_meta, updated_df, 'race_crime_match_cal2',  cal = .2)
updated_df = add_propensity(state_race_crime_cols, 'gender_women', 'gender', ohe_meta, updated_df, 'state_race_crime_match_cal2',  cal = .2)


/usr/local/lib/python3.12/dist-packages/psmpy/psmpy.py:374: UserWarning: Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False
  warnings.warn('Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False')
/usr/local/lib/python3.12/dist-packages/psmpy/psmpy.py:374: UserWarning: Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (effect size). If you do not wish this to be the case please set drop_unmatched=False
  warnings.warn('Some values do not have a match. These are dropped for purposes of establishing a matched dataframe, and subsequent calculations and plots (eff

In [47]:
updated_df

,Unnamed: 0,document_name,Name,State,Race/Ethnicity,gender,race_crime_match_prop_scores,psmpropensity_race_crime_match,state_race_crime_match_prop_scores,psmpropensity_state_race_crime_match,race_crime_match_cal2_prop_scores,psmpropensity_race_crime_match_cal2,state_race_crime_match_cal2_prop_scores,psmpropensity_state_race_crime_match_cal2
0,0,Christie Michelle Scott.txt,Christie Michelle Scott,AL,White,women,0.764350,True,0.756653,True,0.764350,True,0.756653,True
1,1,Heather Keaton.txt,Heather Leavell Keaton,AL,"Native American,White,Black",women,0.792465,True,0.804618,True,0.792465,True,0.804618,True
2,2,Lisa Graham.txt,Lisa Carpenter Graham,AL,White,women,0.624890,True,0.639617,True,0.624890,True,0.639617,True
3,3,Patricia Blackmon.txt,Patricia Blackmon,AL,Black,women,0.565850,True,0.635082,True,0.565850,True,0.635082,True
4,4,Tierra Capri Gobble.txt,Tierra Capri Gobble,AL,White,women,0.700201,True,0.713026,True,0.700201,True,0.713026,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
100,100,Manuel Velez Combined Transcript.txt,"Velez, Manuel",TX,Latino,men,0.069327,False,0.043996,False,0.069327,False,0.043996,False
101,101,Willie Terion Washington Combined Transcript.txt,"Washington, Willie Terion",TX,Black,men,0.374027,True,0.180098,False,0.374027,True,0.180098,False
102,102,Perry Williams Combined Transcript.txt,"Williams, Perry E",TX,Black,men,0.374027,True,0.180098,False,0.374027,True,0.180098,False
103,103,David Leonard Wood Combined Transcript.txt,"Wood, David Leonard",TX,White,men,0.384603,True,0.228538,True,0.384603,True,0.228538,True


In [48]:
updated_df.to_csv(save_path+ '/meta_with_psm_matches_caliper.csv')